# Resonator Bank Implementation Comparison

Benchmarking different implementations of the Resonate algorithm (François, ICMC 2025).

**Test signal:** Log chirp sweeping MIDI piano range (A0=27.5 Hz to C8=4186 Hz)
**Resonator bank:** 88 bins, one per piano key, equal temperament from A440
**Sample rate:** 44100 Hz

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

SR = 44100.0
DURATION = 5.0  # seconds — long enough for stable timing
N_SAMPLES = int(SR * DURATION)
HOP_LENGTH = 256

# Piano range at 10 bins per semitone (120 bins/octave)
BINS_PER_SEMITONE = 10
BINS_PER_OCTAVE = BINS_PER_SEMITONE * 12
FMIN = 27.5   # A0
FMAX = 4186.0 # C8
N_OCTAVES = np.log2(FMAX / FMIN)
N_BINS = int(np.round(N_OCTAVES * BINS_PER_OCTAVE))
FREQUENCIES = FMIN * 2.0 ** (np.arange(N_BINS) / BINS_PER_OCTAVE)

# Alpha heuristic (from original noFFT)
ALPHAS = 1.0 - np.exp(-(1.0 / SR) * FREQUENCIES / np.log10(1.0 + FREQUENCIES))
BETAS = ALPHAS.copy()

# Log chirp: A0 to C8
t = np.linspace(0, DURATION, N_SAMPLES, dtype=np.float64)
f0, f1 = FREQUENCIES[0], FREQUENCIES[-1]
chirp = np.cos(2 * np.pi * f0 * DURATION / np.log(f1 / f0) * (np.exp(t / DURATION * np.log(f1 / f0)) - 1))
chirp_f32 = chirp.astype(np.float32)

print(f"Signal: {N_SAMPLES:,} samples ({DURATION}s @ {SR:.0f} Hz)")
print(f"Bank: {N_BINS} resonators ({BINS_PER_SEMITONE} bins/semitone), {FREQUENCIES[0]:.1f} Hz - {FREQUENCIES[-1]:.1f} Hz")
print(f"Hop: {HOP_LENGTH} samples ({HOP_LENGTH / SR * 1000:.1f} ms)")
print(f"Output frames: {N_SAMPLES // HOP_LENGTH}")

## Reference: noFFT (C++ Accelerate via pybind11)

In [2]:
from noFFT import resonate

def bench(fn, n_runs=5, label=""):
    """Benchmark a function, return (mean_ms, min_ms, result)."""
    # Warmup
    result = fn()
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        result = fn()
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    mean_ms = np.mean(times) * 1000
    min_ms = np.min(times) * 1000
    throughput = N_SAMPLES / np.mean(times)
    rtf = np.mean(times) / DURATION
    print(f"{label}: mean={mean_ms:.2f}ms  min={min_ms:.2f}ms  "
          f"throughput={throughput:,.0f} samples/s  RTF={rtf:.4f}")
    return mean_ms, min_ms, result

def run_nofft():
    return resonate(chirp_f32, SR, 
                    FREQUENCIES.astype(np.float32), 
                    ALPHAS.astype(np.float32),
                    BETAS.astype(np.float32), 
                    HOP_LENGTH)

nofft_ms, _, ref_output = bench(run_nofft, label="noFFT C++ (Accelerate)")

# Reshape to (n_slices, 2*n_bins) split complex and extract powers
ref_complex = ref_output.reshape(-1, 2 * N_BINS)
ref_re = ref_complex[:, :N_BINS]
ref_im = ref_complex[:, N_BINS:]
ref_powers = ref_re**2 + ref_im**2

print(f"Output shape: {ref_powers.shape}")

noFFT C++ (Accelerate): mean=14.80ms  min=14.69ms  throughput=14,894,413 samples/s  RTF=0.0030
Output shape: (861, 88)


In [ ]:
# Visualize reference output
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(ref_powers.T, aspect='auto', origin='lower', 
          extent=[0, DURATION, 0, N_BINS])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Bin')
ax.set_title(f'Reference spectrogram (noFFT C++ Accelerate) — {N_BINS} bins')
plt.colorbar(ax.images[0], ax=ax, label='Power')
plt.tight_layout()
plt.show()

## Naive Python (sin/cos every sample)

In [ ]:
def resonate_naive(signal, sr, frequencies, alphas, betas, hop_length):
    """Naive Python: computes sin/cos every sample. This is the slowest possible version."""
    n_samples = len(signal)
    n_bins = len(frequencies)
    n_slices = n_samples // hop_length
    
    oma = 1.0 - alphas
    omb = 1.0 - betas
    dt = 1.0 / sr
    
    # State
    r_re = np.zeros(n_bins)
    r_im = np.zeros(n_bins)
    rr_re = np.zeros(n_bins)
    rr_im = np.zeros(n_bins)
    
    output_re = np.zeros((n_slices, n_bins))
    output_im = np.zeros((n_slices, n_bins))
    
    omega = -2.0 * np.pi * frequencies
    
    for s in range(n_slices):
        for j in range(hop_length):
            idx = s * hop_length + j
            t = idx * dt
            x = signal[idx]
            
            # Compute phasor from scratch every sample (the naive way)
            z_re = np.cos(omega * t)
            z_im = np.sin(omega * t)
            
            # EWMA accumulation
            r_re = oma * r_re + alphas * x * z_re
            r_im = oma * r_im + alphas * x * z_im
            
            # Output smoothing
            rr_re = omb * rr_re + betas * r_re
            rr_im = omb * rr_im + betas * r_im
        
        output_re[s] = rr_re
        output_im[s] = rr_im
    
    return output_re, output_im

def validate(name, test_powers, ref_powers):
    """Validate against reference: correlation, allclose, max error."""
    corr = np.corrcoef(test_powers.flatten(), ref_powers.flatten())[0, 1]
    max_err = np.max(np.abs(test_powers - ref_powers))
    mean_err = np.mean(np.abs(test_powers - ref_powers))
    # f32 vs f64 precision: use tolerances appropriate for f32 accumulation
    close = np.allclose(test_powers, ref_powers, atol=1e-4, rtol=1e-3)
    print(f"  {name} vs reference:")
    print(f"    correlation={corr:.8f}  max_err={max_err:.2e}  mean_err={mean_err:.2e}  allclose={close}")
    return corr, max_err

# Benchmark on a short signal — Python is SLOW especially at high bin counts
SHORT_DUR = 0.1
short_n = int(SR * SHORT_DUR)
short_chirp = chirp[:short_n]
short_slices = short_n // HOP_LENGTH

def run_naive():
    return resonate_naive(short_chirp, SR, FREQUENCIES, ALPHAS, BETAS, HOP_LENGTH)

naive_ms, _, (naive_re, naive_im) = bench(run_naive, n_runs=1, label=f"Naive Python ({SHORT_DUR}s)")

naive_powers = naive_re**2 + naive_im**2
ref_short = ref_powers[:short_slices]
validate("Naive Python", naive_powers, ref_short)

# Extrapolate to full duration for fair comparison
naive_ms_full = naive_ms * (DURATION / SHORT_DUR)
print(f"Extrapolated to {DURATION}s: {naive_ms_full:.0f}ms")

## Precomputed Phasors (eliminate trig from hot loop)

In [ ]:
def resonate_precomputed(signal, sr, frequencies, alphas, betas, hop_length):
    """Precomputed phasors: sin/cos once at init, complex multiply per sample."""
    n_samples = len(signal)
    n_bins = len(frequencies)
    n_slices = n_samples // hop_length
    
    oma = 1.0 - alphas
    omb = 1.0 - betas
    
    # Precompute phasor multipliers — one trig call total
    angles = -2.0 * np.pi * frequencies / sr
    w_re = np.cos(angles)
    w_im = np.sin(angles)
    
    # Phasor state (starts at 1+0j)
    z_re = np.ones(n_bins)
    z_im = np.zeros(n_bins)
    
    # Resonator state
    r_re = np.zeros(n_bins)
    r_im = np.zeros(n_bins)
    rr_re = np.zeros(n_bins)
    rr_im = np.zeros(n_bins)
    
    output_re = np.zeros((n_slices, n_bins))
    output_im = np.zeros((n_slices, n_bins))
    
    for s in range(n_slices):
        for j in range(hop_length):
            idx = s * hop_length + j
            x = signal[idx]
            
            # EWMA accumulation
            r_re = oma * r_re + alphas * x * z_re
            r_im = oma * r_im + alphas * x * z_im
            
            # Output smoothing
            rr_re = omb * rr_re + betas * r_re
            rr_im = omb * rr_im + betas * r_im
            
            # Phasor rotation (complex multiply, no trig)
            new_zr = z_re * w_re - z_im * w_im
            new_zi = z_re * w_im + z_im * w_re
            z_re = new_zr
            z_im = new_zi
        
        # Stabilize phasors (normalize to unit magnitude)
        mag = np.sqrt(z_re**2 + z_im**2)
        z_re /= mag
        z_im /= mag
        
        output_re[s] = rr_re
        output_im[s] = rr_im
    
    return output_re, output_im

def run_precomputed():
    return resonate_precomputed(short_chirp, SR, FREQUENCIES, ALPHAS, BETAS, HOP_LENGTH)

precomp_ms, _, (precomp_re, precomp_im) = bench(run_precomputed, n_runs=1, label=f"Precomputed phasors ({SHORT_DUR}s)")

precomp_powers = precomp_re**2 + precomp_im**2
validate("Precomputed phasors", precomp_powers, ref_short)
print(f"Speedup vs naive: {naive_ms / precomp_ms:.1f}x")

precomp_ms_full = precomp_ms * (DURATION / SHORT_DUR)
print(f"Extrapolated to {DURATION}s: {precomp_ms_full:.0f}ms")

## noFFT C++ (Accelerate/vDSP, timed fairly from Python)

In [6]:
# Already benchmarked above on the full signal — just reprint
print(f"noFFT C++ (Accelerate): {nofft_ms:.2f}ms for {DURATION}s signal")
print(f"Speedup vs naive (extrapolated): {naive_ms_full / nofft_ms:.1f}x")
print(f"Speedup vs precomputed (extrapolated): {precomp_ms_full / nofft_ms:.1f}x")

noFFT C++ (Accelerate): 14.80ms for 5.0s signal
Speedup vs naive (extrapolated): 106.1x
Speedup vs precomputed (extrapolated): 96.5x


## Rust Scalar (individual Resonators via PyO3, no SIMD)

In [7]:
import resonators

def run_rust_naive():
    return resonators.resonate_naive(
        chirp_f32, SR,
        FREQUENCIES.astype(np.float32),
        ALPHAS.astype(np.float32),
        BETAS.astype(np.float32),
        HOP_LENGTH,
    )

rust_naive_ms, _, rust_naive_output = bench(run_rust_naive, label="Rust scalar (PyO3)")

# Reshape and extract powers (same format as noFFT)
rust_complex = rust_naive_output.reshape(-1, 2 * N_BINS)
rust_re = rust_complex[:, :N_BINS]
rust_im = rust_complex[:, N_BINS:]
rust_powers = rust_re**2 + rust_im**2

validate("Rust scalar", rust_powers, ref_powers)
print(f"Speedup vs naive Python (extrapolated): {naive_ms_full / rust_naive_ms:.1f}x")
print(f"Speedup vs noFFT C++: {nofft_ms / rust_naive_ms:.2f}x")

Rust scalar (PyO3): mean=73.72ms  min=72.33ms  throughput=2,991,030 samples/s  RTF=0.0147
  Rust scalar vs reference:
    correlation=1.00000000  max_err=7.50e-06  mean_err=4.90e-08  allclose=True
Speedup vs naive Python (extrapolated): 21.3x
Speedup vs noFFT C++: 0.20x


## Rust ResonatorBank (SoA layout, no explicit SIMD yet)

In [ ]:
from resonators._resonators import resonate as resonate_bank

def run_rust_bank():
    return resonate_bank(chirp_f32, SR, FREQUENCIES, ALPHAS, BETAS, HOP_LENGTH)

rust_bank_ms, _, rust_bank_output = bench(run_rust_bank, label="Rust ResonatorBank (SoA)")

# Validate
bank_complex = rust_bank_output.reshape(-1, 2 * N_BINS)
bank_re = bank_complex[:, :N_BINS]
bank_im = bank_complex[:, N_BINS:]
bank_powers = bank_re**2 + bank_im**2

validate("Rust ResonatorBank", bank_powers, ref_powers)
print(f"Speedup vs naive Python (extrapolated): {naive_ms_full / rust_bank_ms:.1f}x")
print(f"Speedup vs Rust scalar: {rust_naive_ms / rust_bank_ms:.1f}x")
print(f"vs noFFT C++: {nofft_ms / rust_bank_ms:.2f}x")

## Rust SIMD ResonatorBank (wide f32x8)

In [ ]:
from resonators._resonators import resonate_simd

def run_rust_simd():
    return resonate_simd(chirp_f32, SR, FREQUENCIES, ALPHAS, BETAS, HOP_LENGTH)

rust_simd_ms, _, rust_simd_output = bench(run_rust_simd, label="Rust SIMD (wide f32x8)")

# Validate
simd_complex = rust_simd_output.reshape(-1, 2 * N_BINS)
simd_re = simd_complex[:, :N_BINS]
simd_im = simd_complex[:, N_BINS:]
simd_powers = simd_re**2 + simd_im**2

validate("Rust SIMD", simd_powers, ref_powers)
print(f"Speedup vs naive Python (extrapolated): {naive_ms_full / rust_simd_ms:.1f}x")
print(f"Speedup vs Rust scalar bank: {rust_bank_ms / rust_simd_ms:.1f}x")
print(f"vs noFFT C++: {nofft_ms / rust_simd_ms:.2f}x")

## Summary

In [ ]:
# Collect all results (extrapolate short benchmarks to full duration)
results = {
    'Naive Python': naive_ms_full,
    'Precomputed Phasors': precomp_ms_full,
    'noFFT C++ (Accelerate)': nofft_ms,
    'Rust scalar (PyO3)': rust_naive_ms,
    'Rust ResonatorBank (SoA)': rust_bank_ms,
    'Rust SIMD (wide f32x8)': rust_simd_ms,
}

fig, ax = plt.subplots(figsize=(10, 5))

names = list(results.keys())
times = list(results.values())
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd', '#8c564b']
bars = ax.barh(names, times, color=colors[:len(names)])
ax.set_xlabel('Time (ms)')
ax.set_title(f'Time to process {DURATION}s of audio ({N_BINS} resonators, hop={HOP_LENGTH})')
ax.set_xscale('log')
for bar, t in zip(bars, times):
    ax.text(t * 1.1, bar.get_y() + bar.get_height()/2, f'{t:.1f}ms', va='center')

plt.tight_layout()
plt.show()

# Summary table
baseline = times[0]
print(f"\n{'Implementation':<30} {'Time (ms)':>10} {'Throughput':>15} {'RTF':>8} {'Speedup':>8}")
print("-" * 75)
for name, ms in results.items():
    tp = N_SAMPLES / (ms / 1000)
    rtf = (ms / 1000) / DURATION
    speedup = baseline / ms
    print(f"{name:<30} {ms:>10.1f} {tp:>12,.0f}/s {rtf:>8.4f} {speedup:>7.0f}x")